In [2]:
import numpy as np
import yfinance as yf
import pandas as pd
import matplotlib.pyplot as plt
from arch import arch_model
import pyvinecopulib as pv
from scipy.stats import genpareto
import os
from dataclasses import dataclass, field, asdict
from typing import Optional, Sequence, Union, List, Dict, Any, Tuple
import numpy as np
import cvxpy as cp
from scipy.spatial.distance import squareform
from scipy.cluster.hierarchy import linkage, fcluster, leaves_list
from sklearn.covariance import LedoitWolf

In [ ]:
@dataclass
class Universe:
  Bonds:List[str]
  ManagedFutures:List[str]
  Commodities:List[str]
  High_Beta:List[str]
  High_Yield:List[str]
  Sat_Defensive:List[str]

In [6]:
class DataStore:
  def __init__(self, debug:bool=False, **kwargs):
    super().__init__(
      debug=debug,
      **kwargs
    )
    self.debug = debug

  def _get_data(
      self,
      universe:dict,
      start:str,
      end:str,
      interval:str="1d",
      benchmark:str="^GSPC"
  ):
    tickers_raw = list(asdict(universe).values())
    tmp = []
    for t in tickers_raw:
      if isinstance(t, list):
        tmp.extend(t)
      elif isinstance(t, str):
        tmp.append(t)

      else:
        print(f"Warning: Skipping {t} | type: {type(t)} ")

    tickers_clean = list(set(tmp))

    self.benchmark_ticker = benchmark
    df_path = f"portfolio_{start}_{end}.parquet"

    if not os.path.exists(df_path):
      if benchmark not in tickers_clean:
        tickers_clean.append(benchmark)

      df = yf.download(tickers_clean, start, end, interval)["Close"]

      df.to_parquet(df_path)

    else:
      df = pd.read_parquet(df_path)

    bench_data = df["^GSPC"]
    data_raw = df.drop(columns=["^GSPC"])

    benchmark = bench_data.pct_change().dropna()
    self.universe = data_raw.columns

    return data_raw, benchmark

  def plot_data(self):
    (np.cumsum(self.returns_raw * 100, axis=0) + 100).plot(figsize=(15, 10))
    plt.show()

  def plot_benchmark(self):
    (np.cumsum(self.benchmark * 100, axis=0) + 100).plot(figsize=(15, 10))
    plt.show()

In [3]:
class GARCHEVTCOPULA:
  def __init__(self, sim_params, debug:bool=False, **kwargs):
    self.sim_params = sim_params
    self.debug = debug
    self.copula = None
    self.best_models = None
    self.best_fits = None
    self.best_params = None
    self.scale_factor = 100

  def fit_model(self, r):
    best_bic = np.inf
    best_fit = None
    best_model = None
    r_scaled = r * self.scale_factor

    model = arch_model(
      r_scaled,
      mean="Constant",
      vol="GARCH",
      p=1,
      o=1,
      q=1,
      dist="Studentst",
    )

    fit = model.fit(disp='off')

    if fit.bic < best_bic:
        best_bic = fit.bic
        best_fit = fit
        best_model = model

    return best_bic, best_fit, best_model

  def fit_ar_garch(self, returns):
    print("------------- Fitting AR-GARCH -------------")
    filtered_resid = pd.DataFrame(index=returns.index, columns=returns.columns)
    self.best_models = {}
    self.best_fits = {}
    self.best_params = {}

    for ticker in returns.columns:
      r = returns[ticker].dropna()

      best_bic, best_fit, best_model = self.fit_model(r)

      z_resid = best_fit.std_resid

      filtered_resid[ticker] = z_resid
      self.best_models[ticker] = best_model
      self.best_fits[ticker] = best_fit

    return filtered_resid.dropna()

  def get_pseudo_observations(self, residuals):
    print("------------ Uniform Residuals -------------")
    uniform_resid = pd.DataFrame(
      index=residuals.index,
      columns=residuals.columns
    )

    self.margin_params = {}

    for ticker in residuals.columns:
      r = residuals[ticker].to_numpy()

      n = len(r)

      q_high = self.sim_params.quantile
      q_low = 1.0 - self.sim_params.quantile

      u_upper = np.percentile(r, q_high * 100)
      u_lower = np.percentile(r, q_low * 100)

      upper_tail = r[r > u_upper]
      lower_tail = r[r < u_lower]

      c_U, _, scale_U = genpareto.fit(upper_tail - u_upper, floc=0)
      c_L, _, scale_L = genpareto.fit(u_lower - lower_tail, floc=0)

      p_l = np.mean(r <= u_lower)
      p_u = np.mean(r >= u_upper)

      u_transformed = np.zeros_like(r, dtype=float)

      low_mask = r < u_lower
      if np.any(low_mask):
        u_transformed[low_mask] = p_l * (
          1.0 \
          - genpareto.cdf(
              u_lower - r[low_mask],
              c_L,
              loc=0,
              scale=scale_L
            )
          )

      high_mask = r > u_upper
      if np.any(high_mask):
          u_transformed[high_mask] = (1.0 - p_u) \
          + p_u * genpareto.cdf(
              r[high_mask] - u_upper,
              c_U,
              loc=0,
              scale=scale_U
            )

      body_mask = (~low_mask) & (~high_mask)
      if np.any(body_mask):
        body_r = r[body_mask]
        ranks = pd.Series(body_r).rank(method='average').to_numpy()
        u_transformed[body_mask] = p_l + (1.0 - p_l - p_u) \
                                    * (ranks / (len(body_r) + 1))

      uniform_resid[ticker] = u_transformed

      self.margin_params[ticker] = {
          'u_upper': u_upper, 'u_lower': u_lower,
          'c_U': c_U, 'scale_U': scale_U,
          'c_L': c_L, 'scale_L': scale_L,
          'p_l': p_l, 'p_u': p_u,
          'body_data': r[(r >= u_lower) & (r <= u_upper)]
      }

    return uniform_resid

  def inverse_semi_parametric_cdf(self, uniform_samples):
    n_paths, n_steps, n_assets = uniform_samples.shape
    simulated_residuals = np.zeros_like(uniform_samples)
    tickers = list(self.margin_params.keys())

    for i, ticker in enumerate(tickers):
        u_col = uniform_samples[:, :, i]
        params = self.margin_params[ticker]

        p_l = params["p_l"]
        p_u = params["p_u"]

        real_col = np.zeros_like(u_col)

        low_mask = u_col < p_l
        if np.any(low_mask):
            real_col[low_mask] = (
                params["u_lower"]
                - genpareto.ppf(
                    1.0 - u_col[low_mask] / p_l,
                    params["c_L"],
                    loc=0,
                    scale=params["scale_L"]
                )
            )

        high_mask = u_col > (1.0 - p_u)
        if np.any(high_mask):
            real_col[high_mask] = (
                params["u_upper"]
                + genpareto.ppf(
                    (u_col[high_mask] - (1.0 - p_u)) / p_u,
                    params["c_U"],
                    loc=0,
                    scale=params["scale_U"]
                )
            )

        body_mask = (~low_mask) & (~high_mask)
        if np.any(body_mask):
            body_percentiles = (u_col[body_mask] - p_l) / (1.0 - p_u - p_l) * 100
            body_percentiles = np.clip(body_percentiles, 0, 100)
            real_col[body_mask] = np.percentile(params['body_data'], body_percentiles)

        simulated_residuals[:, :, i] = real_col

    return simulated_residuals

  def _fit_vine_copula(self, residuals):
    print("-------------- Fitting Copula --------------")

  def fit(self, returns, save=True):
    residuals = self.fit_ar_garch(returns)
    self.cols = residuals.columns
    self._fit_vine_copula(residuals)

  def generate_sample(self):
    pass



In [ ]:
class CVaREngine(GARCHEVTCOPULA):
  def __init__(self, cvar_engine_params, debug:bool=False, **kwargs):
    self.debug = debug
    self.cvar_params = cvar_engine_params

  def fit_garch_evt_copula(self, returns):
    pass
  
